In [3]:
from collections import OrderedDict
from typing import Any, Dict, List, Optional, Union


In [4]:

# ---- Helpers ----------------------------------------------------------------

def _unwrap_value(v: Any) -> Any:
    """Return v['value'] if v looks like a W&B-wrapped value; else v."""
    if isinstance(v, dict) and "value" in v and len(v) <= 3:
        return v["value"]
    return v

def _maybe_bool_str_to_bool(v: Any) -> Any:
    if isinstance(v, str):
        s = v.strip().lower()
        if s == "true": return True
        if s == "false": return False
        if s == "none": return None
    return v

def _maybe_numeric_str(v: Any) -> Any:
    if isinstance(v, str):
        s = v.strip()
        # Don't coerce strings that obviously aren't pure numerics (e.g., comments)
        if any(ch.isalpha() for ch in s.replace(".", "").replace("-", "").replace("+", "").replace("e", "", 1)):
            return v
        try:
            if any(ch in s for ch in (".", "e", "E")):
                return float(s)
            return int(s)
        except ValueError:
            return v
    return v

def _parse_int_list(x: Any) -> Optional[List[int]]:
    if isinstance(x, (list, tuple)):
        try:
            return [int(v) for v in x]
        except Exception:
            return None
    if isinstance(x, str):
        parts = [p.strip() for p in x.split(",")]
        try:
            return [int(p) for p in parts if p]
        except Exception:
            return None
    return None

def _get(cfg: Dict[str, Any], key: str, default: Any = None) -> Any:
    raw = cfg.get(key, default)
    val = _unwrap_value(raw)
    # Only coerce when it's a simple scalar; lists/dicts should pass through
    if isinstance(val, (list, dict, tuple)):
        return val
    val = _maybe_bool_str_to_bool(val)
    val = _maybe_numeric_str(val)
    return val

def _format_bool(v: bool) -> str:
    return "True" if v else "False"

def _format_simple(v: Any) -> str:
    if isinstance(v, bool):
        return _format_bool(v)
    if isinstance(v, float):
        return repr(v)  # preserve precision
    return str(v)

def _format_list_inline(lst: List[Any]) -> str:
    return "[" + ", ".join(_format_simple(x) for x in lst) + "]"

def _format_delay_list_block(lst: List[Any]) -> str:
    lines = ["["]
    for x in lst:
        lines.append("        " + _format_simple(int(x)) + ",")
    lines.append("        ]")
    return "\n".join(lines)

# ---- Main renderer -----------------------------------------------------------

SECTIONS = OrderedDict({
    "power settings": [
        "use_constant_power",
        "constant_power_per_processor",
        "procs_per_node",
        "idle_power",
        "carbon_year",
        "custom_intensity",   # may contain an inline comment string — we pass through as-is
        "user_ci",
    ],
    "architecture": [
        "green_forecast_length",
        "max_queue_size",
        "run_win_length",
        "delay_time_list",
        "max_wait_n_jobs",
        "job_feature",
        "run_feature",
        "green_feature_pr_timeslot",
        "green_feature_constant",
    ],
    "training": [
        "episode_length",
        "gamma",
        "gae_lambda",
        "batch_size",
        "seed",
        "n_epochs",
        "pi_nn",
        "vf_nn",
        "n_steps",
        "total_timesteps",
        "ent_coef",
        "learning_rate",
        "clip_range",
        "vf_coef",
        "clip_range_vf",
        "normalize_advantage",
        "max_grad_norm",
        "n_envs",
        "sweep_seeds",
        "validation_freq",
        "validation_episodes",
    ],
    "reward": [
        "base_line_wait_carbon_penality",
        "eta",
        "reward_type",
        "abs_carbon_reward_clip",
        "wait_reward_clip",
        "wait_reward_booster",
        "carbon_reward_booster",
    ],
    "normalization constants": [
        "max_power",
        "max_green",
        "max_wait_time",
        "max_run_time",
        "max_requested_processors",
    ],
})

def render_config_txt(config: Dict[str, Any]) -> str:
    """
    Convert a W&B-like config dict (values under {"value": ...}) into the desired
    plain-text config with sections.
    """
    lines: List[str] = []

    # Pre-extract some special keys that need parsing/formatting:
    # pi_nn / vf_nn may be comma strings -> [ints]
    pi_nn_val = _get(config, "pi_nn")
    vf_nn_val = _get(config, "vf_nn")
    pi_nn_list = _parse_int_list(pi_nn_val) if pi_nn_val is not None else None
    vf_nn_list = _parse_int_list(vf_nn_val) if vf_nn_val is not None else None

    # delay_time_list must be a pretty-printed block
    delay_list = _get(config, "delay_time_list")
    if isinstance(delay_list, (list, tuple)):
        try:
            delay_list = [int(x) for x in delay_list]
        except Exception:
            pass  # leave as-is if weird

    # sweep_seeds should render inline [a, b, c]
    sweep_seeds = _get(config, "sweep_seeds")

    for section, keys in SECTIONS.items():
        lines.append(f"[{section}]")
        for k in keys:
            v = None
            if k == "pi_nn":
                v = pi_nn_list if pi_nn_list is not None else pi_nn_val
                if v is None:
                    continue
                if isinstance(v, list):
                    lines.append(f"{k} = {_format_list_inline(v)}")
                else:
                    # fallback: print as-is
                    lines.append(f"{k} = {v}")
                continue

            if k == "vf_nn":
                v = vf_nn_list if vf_nn_list is not None else vf_nn_val
                if v is None:
                    continue
                if isinstance(v, list):
                    lines.append(f"{k} = {_format_list_inline(v)}")
                else:
                    lines.append(f"{k} = {v}")
                continue

            if k == "delay_time_list":
                v = delay_list
                if v is None:
                    continue
                if isinstance(v, list):
                    lines.append(f"{k} = " + _format_delay_list_block(v))
                else:
                    # if somehow a string got through, just print it
                    lines.append(f"{k} = {v}")
                continue

            if k == "sweep_seeds":
                v = sweep_seeds
                if v is None:
                    continue
                if isinstance(v, (list, tuple)):
                    lines.append(f"{k} = {_format_list_inline(list(v))}")
                else:
                    lines.append(f"{k} = {v}")
                continue

            # Normal path
            v = _get(config, k, default=None)
            if v is None:
                continue  # skip missing keys

            # Keep inline comments on strings (e.g., custom_intensity)
            if isinstance(v, str) and "##" in v:
                lines.append(f"{k} = {v}")
                continue

            if isinstance(v, bool):
                lines.append(f"{k} = {_format_bool(v)}")
            else:
                lines.append(f"{k} = {_format_simple(v)}")
        lines.append("")  # blank line between sections

    return "\n".join(lines).rstrip()  # trim trailing newline




In [11]:
cfg ={
  "env": {
    "value": "<stable_baselines3.common.vec_env.vec_normalize.VecNormalize object at 0x7fa070f15cd0>"
  },
  "eta": {
    "value": 0.001
  },
  "algo": {
    "value": "MaskablePPO"
  },
  "seed": {
    "value": 6
  },
  "gamma": {
    "value": 0.9865612513398736
  },
  "pi_nn": {
    "value": "256,256"
  },
  "vf_nn": {
    "value": "2048,1024"
  },
  "_wandb": {
    "value": {
      "e": {
        "myxyr5t3plw30ancbdrn9g51mg0njf45": {
          "os": "Linux-4.18.0-553.40.1.el8_10.x86_64-x86_64-with-glibc2.28",
          "git": {
            "commit": "654d1acaeeec05de1224f55dc84c902f9119a935",
            "remote": "https://github.com/dadyownes15/green-hpc-scheduler.git"
          },
          "args": [
            "--sweep",
            "sweep_config.yaml",
            "--count",
            "50",
            "--eta",
            "0.001"
          ],
          "disk": {
            "/": {
              "used": "30019633152",
              "total": "161008844800"
            }
          },
          "host": "hendrixgpu26fl.unicph.domain",
          "root": "/home/xdg178/green-hpc-scheduler",
          "email": "mikkeljanusdahl@gmail.com",
          "slurm": {
            "conf": "/var/spool/slurmd/conf-cache/slurm.conf",
            "gtids": "0",
            "jobid": "4664",
            "job_id": "4664",
            "nnodes": "1",
            "nodeid": "0",
            "procid": "0",
            "job_gid": "100",
            "job_qos": "normal",
            "job_uid": "495620150",
            "localid": "0",
            "job_name": "eta_sweep",
            "job_user": "xdg178",
            "nodelist": "hendrixgpu26fl",
            "task_pid": "3656833",
            "submit_dir": "/home/xdg178/green-hpc-scheduler",
            "job_account": "ml",
            "mem_per_cpu": "4096",
            "submit_host": "hendrixgate03fl.unicph.domain",
            "array_job_id": "4659",
            "cluster_name": "dicompute",
            "cpus_on_node": "16",
            "job_nodelist": "hendrixgpu26fl",
            "node_aliases": "(null)",
            "prio_process": "0",
            "array_task_id": "5",
            "cpus_per_task": "16",
            "job_num_nodes": "1",
            "job_partition": "gpu",
            "topology_addr": "hendrixgpu26fl",
            "array_task_max": "6",
            "array_task_min": "1",
            "tasks_per_node": "1",
            "array_task_step": "1",
            "working_cluster": "dicompute:hendrixhead01fl.unicph.domain:6817:9216:109",
            "array_task_count": "6",
            "job_cpus_per_node": "16",
            "topology_addr_pattern": "node"
          },
          "memory": {
            "total": "269411860480"
          },
          "python": "CPython 3.12.11",
          "program": "/home/xdg178/green-hpc-scheduler/sweep_multi_env.py",
          "codePath": "sweep_multi_env.py",
          "writerId": "myxyr5t3plw30ancbdrn9g51mg0njf45",
          "cpu_count": 16,
          "startedAt": "2025-10-19T04:53:40.229724Z",
          "executable": "/home/xdg178/green-hpc-scheduler/venv/bin/python",
          "codePathLocal": "sweep_multi_env.py",
          "cpu_count_logical": 32
        }
      },
      "m": [],
      "t": {
        "1": [
          1
        ],
        "2": [
          1
        ],
        "3": [
          2,
          14,
          17,
          22,
          62
        ],
        "4": "3.12.11",
        "5": "0.22.0",
        "12": "0.22.0",
        "13": "linux-x86_64"
      },
      "cli_version": "0.22.0",
      "python_version": "3.12.11"
    }
  },
  "device": {
    "value": "cpu"
  },
  "n_envs": {
    "value": 16
  },
  "policy": {
    "value": "MaskableActorCriticPolicy(\n  (features_extractor): FlattenExtractor(\n    (flatten): Flatten(start_dim=1, end_dim=-1)\n  )\n  (pi_features_extractor): FlattenExtractor(\n    (flatten): Flatten(start_dim=1, end_dim=-1)\n  )\n  (vf_features_extractor): FlattenExtractor(\n    (flatten): Flatten(start_dim=1, end_dim=-1)\n  )\n  (mlp_extractor): MlpExtractor(\n    (policy_net): Sequential(\n      (0): Linear(in_features=1441, out_features=256, bias=True)\n      (1): Tanh()\n      (2): Linear(in_features=256, out_features=256, bias=True)\n      (3): Tanh()\n    )\n    (value_net): Sequential(\n      (0): Linear(in_features=1441, out_features=2048, bias=True)\n      (1): Tanh()\n      (2): Linear(in_features=2048, out_features=1024, bias=True)\n      (3): Tanh()\n    )\n  )\n  (action_net): Linear(in_features=256, out_features=266, bias=True)\n  (value_net): Linear(in_features=1024, out_features=1, bias=True)\n)"
  },
  "_logger": {
    "value": "<stable_baselines3.common.logger.Logger object at 0x7fa070f174a0>"
  },
  "n_steps": {
    "value": 65536
  },
  "use_sde": {
    "value": "False"
  },
  "user_ci": {
    "value": "disabled"
  },
  "verbose": {
    "value": 1
  },
  "vf_coef": {
    "value": 0.5
  },
  "ent_coef": {
    "value": 0.01
  },
  "n_epochs": {
    "value": 1
  },
  "_last_obs": {
    "value": "[[ 0.         -0.5257845  -0.40178677 ... -0.9039211  -0.85962754\n  -1.0866107 ]\n [ 0.         -0.526136   -0.32182762 ... -0.34740177 -0.13056998\n  -0.20066704]\n [ 0.         -0.52789366 -0.00199099 ...  0.26443002  0.19049248\n   0.14206253]\n ...\n [ 0.          0.62898695 -0.40178677 ...  0.52326316  0.7093422\n   0.61365426]\n [ 0.          0.18934186  0.42445788 ... -0.00240582 -0.07984779\n  -0.20386985]\n [ 0.         -0.52742493 -0.37513372 ...  0.999109    1.0023406\n   1.0815779 ]]"
  },
  "max_green": {
    "value": 19000
  },
  "max_power": {
    "value": 19000
  },
  "target_kl": {
    "value": "None"
  },
  "_n_updates": {
    "value": 0
  },
  "batch_size": {
    "value": 2048
  },
  "clip_range": {
    "value": 0.1
  },
  "gae_lambda": {
    "value": 0.9699230977202816
  },
  "idle_power": {
    "value": 15
  },
  "start_time": {
    "value": 1760849637968200700
  },
  "carbon_year": {
    "value": 2021
  },
  "job_feature": {
    "value": 5
  },
  "lr_schedule": {
    "value": "FloatSchedule(ConstantSchedule(val=0.0006596282056543944))"
  },
  "reward_type": {
    "value": "delay_queue_penalty_abs_ems"
  },
  "run_feature": {
    "value": 2
  },
  "sweep_seeds": {
    "value": [
      0,
      1,
      2
    ]
  },
  "_episode_num": {
    "value": 0
  },
  "action_noise": {
    "value": "None"
  },
  "action_space": {
    "value": "Discrete(266)"
  },
  "max_run_time": {
    "value": 162754
  },
  "policy_class": {
    "value": "<class 'sb3_contrib.common.maskable.policies.MaskableActorCriticPolicy'>"
  },
  "clip_range_vf": {
    "value": 0.05
  },
  "learning_rate": {
    "value": 0.0006596282056543944
  },
  "max_grad_norm": {
    "value": 0.8
  },
  "max_wait_time": {
    "value": 20000
  },
  "num_timesteps": {
    "value": 0
  },
  "policy_kwargs": {
    "value": "{'net_arch': {'pi': [256, 256], 'vf': [2048, 1024]}}"
  },
  "_custom_logger": {
    "value": "False"
  },
  "ep_info_buffer": {
    "value": "deque([], maxlen=100)"
  },
  "episode_length": {
    "value": 32
  },
  "max_queue_size": {
    "value": 256
  },
  "procs_per_node": {
    "value": 1
  },
  "rollout_buffer": {
    "value": "<sb3_contrib.common.maskable.buffers.MaskableRolloutBuffer object at 0x7fa070f170b0>"
  },
  "run_win_length": {
    "value": 65
  },
  "delay_time_list": {
    "value": [
      300,
      600,
      1200,
      1800,
      2400,
      3600,
      7200,
      14400
    ]
  },
  "max_wait_n_jobs": {
    "value": 2
  },
  "sde_sample_freq": {
    "value": -1
  },
  "tensorboard_log": {
    "value": "None"
  },
  "total_timesteps": {
    "value": 524288
  },
  "validation_freq": {
    "value": 262144
  },
  "_total_timesteps": {
    "value": 524288
  },
  "custom_intensity": {
    "value": "False ## If true it utilizes the a custom intensity, else it use real data"
  },
  "wait_reward_clip": {
    "value": 20
  },
  "ep_success_buffer": {
    "value": "deque([], maxlen=100)"
  },
  "observation_space": {
    "value": "Box(0.0, 1.0, (1441,), float32)"
  },
  "_last_original_obs": {
    "value": "[[ 0.         -0.5257845  -0.40178677 ... -0.9039211  -0.85962754\n  -1.0866107 ]\n [ 0.         -0.526136   -0.32182762 ... -0.34740177 -0.13056998\n  -0.20066704]\n [ 0.         -0.52789366 -0.00199099 ...  0.26443002  0.19049248\n   0.14206253]\n ...\n [ 0.          0.62898695 -0.40178677 ...  0.52326316  0.7093422\n   0.61365426]\n [ 0.          0.18934186  0.42445788 ... -0.00240582 -0.07984779\n  -0.20386985]\n [ 0.         -0.52742493 -0.37513372 ...  0.999109    1.0023406\n   1.0815779 ]]"
  },
  "_stats_window_size": {
    "value": 100
  },
  "_vec_normalize_env": {
    "value": "<stable_baselines3.common.vec_env.vec_normalize.VecNormalize object at 0x7fa070f15cd0>"
  },
  "use_constant_power": {
    "value": True
  },
  "normalize_advantage": {
    "value": True
  },
  "validation_episodes": {
    "value": 1
  },
  "wait_reward_booster": {
    "value": 2
  },
  "_last_episode_starts": {
    "value": "[ True  True  True  True  True  True  True  True]"
  },
  "rollout_buffer_class": {
    "value": "<class 'sb3_contrib.common.maskable.buffers.MaskableRolloutBuffer'>"
  },
  "carbon_reward_booster": {
    "value": 0.01
  },
  "green_forecast_length": {
    "value": 24
  },
  "rollout_buffer_kwargs": {
    "value": "{}"
  },
  "abs_carbon_reward_clip": {
    "value": 10
  },
  "delay_time_list_length": {
    "value": 8
  },
  "green_feature_constant": {
    "value": 8
  },
  "_num_timesteps_at_start": {
    "value": 0
  },
  "max_requested_processors": {
    "value": 256
  },
  "green_feature_pr_timeslot": {
    "value": 1
  },
  "_current_progress_remaining": {
    "value": 1
  },
  "constant_power_per_processor": {
    "value": 500
  },
  "base_line_wait_carbon_penality": {
    "value": 0.01
  }
}

In [9]:
print(render_config_txt(cfg))

[power settings]
use_constant_power = True
constant_power_per_processor = 500
procs_per_node = 1
idle_power = 15
carbon_year = 2021
custom_intensity = False ## If true it utilizes the a custom intensity, else it use real data
user_ci = disabled

[architecture]
green_forecast_length = 24
max_queue_size = 256
run_win_length = 65
delay_time_list = [
        300,
        600,
        1200,
        1800,
        2400,
        3600,
        7200,
        14400,
        ]
max_wait_n_jobs = 2
job_feature = 5
run_feature = 2
green_feature_pr_timeslot = 1
green_feature_constant = 8

[training]
episode_length = 1024
gamma = 0.971246506345966
gae_lambda = 0.98253867295831
batch_size = 4096
seed = 6
n_epochs = 4
pi_nn = [512, 512]
vf_nn = [2048, 1024]
n_steps = 8192
total_timesteps = 524288
ent_coef = 0.005
learning_rate = 0.00012489194207864325
clip_range = 0.1
vf_coef = 0.5
clip_range_vf = 0.2
normalize_advantage = True
max_grad_norm = 0.8
n_envs = 16
sweep_seeds = [0, 1, 2]
validation_freq = 262